# GiftVoice - live voice demo

A real-time voice gift concierge: talk to it, and it searches a 200-product catalog, shows cards,
handles gift wrap, delivery deadlines and a test checkout.
Code: [github.com/usamahassan965/GiftVoice](https://github.com/usamahassan965/GiftVoice)

This notebook runs the whole agent on Colab's free CPU and gives you a public HTTPS link you can
open (or share) for as long as the notebook stays running.

**Before you start** you need two free API keys:

| Key | Where to get it |
|---|---|
| `GROQ_API_KEY` | [console.groq.com/keys](https://console.groq.com/keys) - speech-to-text |
| `GOOGLE_API_KEY` | [aistudio.google.com/apikey](https://aistudio.google.com/apikey) - the LLM |

Add them in Colab's **key icon** in the left sidebar (*Secrets*), with those exact names, and turn on
*Notebook access* for both. They stay in your Google account - the notebook never prints them.

Then run the cells in order (Runtime -> Run all works too). Total start-up is about 5 minutes.


## 1. Keys


In [2]:
from google.colab import userdata
import os

REQUIRED = ['GROQ_API_KEY', 'GOOGLE_API_KEY']
OPTIONAL = ['STRIPE_SECRET_KEY']  # sk_test_... only, for the checkout step

missing = []
for name in REQUIRED + OPTIONAL:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        print(f'{name}: loaded')
    elif name in REQUIRED:
        missing.append(name)
    else:
        print(f'{name}: not set (optional)')

if missing:
    raise SystemExit(
        'Missing secret(s): ' + ', '.join(missing) +
        "\nOpen the key icon in the left sidebar, add them, enable 'Notebook access', then re-run.")


GROQ_API_KEY: loaded
GOOGLE_API_KEY: loaded
STRIPE_SECRET_KEY: not set (optional)


## 2. Download the code and install


In [ ]:
%%bash
set -e
if [ ! -d /content/GiftVoice ]; then
  git clone --depth 1 -q https://github.com/usamahassan965/GiftVoice.git /content/GiftVoice
fi
cd /content/GiftVoice
git pull -q || true

# The product catalog (SQLite + Chroma vectors + 214 photos) ships zipped in the repo.
python3 backend/scripts/unpack_catalog.py

# An isolated env keeps Colab's own preinstalled packages out of the way.
pip install -q uv
uv venv -q --python 3.12 /content/venv
uv pip install -q --python /content/venv/bin/python -r backend/requirements.txt \
  --extra-index-url https://download.pytorch.org/whl/cpu --index-strategy unsafe-best-match
echo 'python deps ready'


## 3. Build the browser UI

The page is a Next.js static export, served by the same process as the agent, so one link is enough.


In [ ]:
%%bash
set -e
cd /content/GiftVoice/frontend
if [ -f out/index.html ]; then echo 'already built'; exit 0; fi

if ! command -v node >/dev/null; then
  curl -sL https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz | tar -xJ -C /opt
  export PATH=/opt/node-v22.14.0-linux-x64/bin:$PATH
fi
npm ci --silent --no-audit --no-fund
NEXT_EXPORT=1 npm run build
echo 'UI built'


## 4. Start the agent and open the public link

This starts the server, then a Cloudflare quick tunnel that gives it an HTTPS address (browsers only
allow microphone access over HTTPS). Keep this cell running while you use the demo.


In [ ]:
import os, re, subprocess, urllib.request

REPO = '/content/GiftVoice'
PY = '/content/venv/bin/python'

if not os.path.exists('/usr/local/bin/cloudflared'):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '/usr/local/bin/cloudflared')
    os.chmod('/usr/local/bin/cloudflared', 0o755)

# The tunnel first, so the app knows its own public URL (Stripe redirects use it).
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in tunnel.stdout:
    found = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if found:
        public_url = found.group(0)
        break
if not public_url:
    raise SystemExit('Could not start the tunnel - re-run this cell.')

env = {**os.environ,
       'TRANSPORT': 'websocket',
       'TTS_ENGINE': 'edge',
       'FRONTEND_URL': public_url,
       'MAX_CONCURRENT_SESSIONS': '2',
       'SESSION_TIMEOUT_SECS': '420',
       'PYTHONUNBUFFERED': '1'}

server = subprocess.Popen(
    [PY, '-m', 'uvicorn', 'app.main:app', '--app-dir', 'backend', '--host', '0.0.0.0', '--port', '7860'],
    cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

print('Loading the search models (about a minute the first time)...')
for line in server.stdout:
    print(line, end='')
    if 'Warm-up finished' in line or 'Application startup complete' in line:
        break

print('\n' + '=' * 70)
print('  Live demo:', public_url)
print('  Allow the microphone, press Start, and say')
print('  "I need a gift for my mum, she loves gardening, under 50 dollars".')
print('=' * 70 + '\n')

try:
    for line in server.stdout:
        print(line, end='')
except KeyboardInterrupt:
    server.terminate(); tunnel.terminate()
    print('stopped')


### Notes

* The link dies when this notebook stops, and a new run gets a new address.
* Free-tier limits: two people at a time, seven minutes per conversation.
* Everything here runs on free quotas (Groq, Google AI Studio, edge-tts), so under load the answers
  can slow down or fall back to the second model.
* Checkout runs in Stripe **test** mode - card `4242 4242 4242 4242` - and no money moves. Without a
  Stripe key the app uses a mock checkout page instead.
